# Cubic MoC (Fm3̄m) — sequential Gaussian-process Bayesian optimization

### The loop
1. Edit `RESULTS` in section 1 — add the run you just finished.
2. Run all cells.
3. Section 9 prints one procedure.
4. Perform it, refine the XRD, go to step 1.

No LLM, no embeddings, no API keys. numpy + scipy + pandas only.

### What the model is
Six synthesis parameters in, MoC weight fraction out. Matérn-5/2 covariance with one
lengthscale per parameter. Observation noise is **fixed** from your Rietveld esds
instead of fitted, because a handful of points cannot separate signal variance from
noise variance. Hyperparameters are **integrated out** by MCMC instead of point-
estimated — at small n that is the single biggest cause of overconfident error bars,
and overconfident error bars make the acquisition stop exploring too early.


## 0. Setup and settings

In [1]:
import os                       # file-existence checks for the dataset path
import textwrap                 # wraps the procedure text for printing in section 9
import numpy as np              # all the linear algebra
import pandas as pd             # the design space and campaign log
from scipy.stats import norm    # normal CDF/PDF, used by expected improvement

pd.set_option("display.width", 200)                    # stop pandas wrapping wide tables
np.set_printoptions(suppress=True, linewidth=160)      # no scientific notation in arrays

DATA = "./dataset/Metal_Sucrose_DesignSpace_fr_T1_encoded.xlsx"
for alt in [DATA,
            "./Metal_Sucrose_DesignSpace_fr_T1_encoded.xlsx"]:
    if os.path.exists(alt):        # take the first path that actually exists,
        DATA = alt                 # so the notebook runs from either directory layout
        break
print("dataset:", DATA)

# ── campaign settings ─────────────────────────────────────────────────────────

GOF_POWER = 1.0
# Rietveld esds are multiplied by GOF**GOF_POWER whenever GOF > 1. GOF = sqrt(chi^2),
# so GOF = 1 means the fit is as good as counting statistics allow and GOF = 10 means
# the structural model does not explain the pattern. 1.0 is the conventional chi-squared
# scaling and downweights a GOF-10 refinement 10x. Use 0.5 if you believe your high GOF
# comes from peak-shape or background misfit rather than bad phase quantification.
# Use 0.0 to trust the raw esds as reported.

SIGMA_FLOOR = 0.5
# wt%. A hard minimum on the uncertainty of any weight fraction. Refinements sometimes
# report esd = 0.00 for a phase that was excluded; a zero-noise observation would be
# treated as infinitely precise and would dominate the entire fit.

DELTA = 0.65
# wt%. 0.65 x the ~1 wt% Rietveld detection limit (Martin-Fernandez et al. 2003).
# Used to pad the logit transform away from its singularities at exactly 0 and 100 wt%,
# so a run that reported 0.0 wt% MoC does not produce log(0) = -infinity.

EI_XI = 0.01
# Exploration nudge inside expected improvement: the model must beat the incumbent by
# this much (in standardised units) before the improvement counts. Larger = more
# exploratory. Only used once the campaign switches to EI.

EI_AFTER = 10
# Number of measured runs required before switching from exploration to expected
# improvement. Below this the posterior variance is largest at the corners of the design
# box, so EI chases regions it knows nothing about rather than regions that look good.

RANDOM_SEED = 616               # fixes the MCMC chain so re-running gives the same answer

# ── column names ──────────────────────────────────────────────────────────────
# The six synthesis parameters, in the order they appear in the RESULTS tuples below.
DESIGN_KEYS = ["Carburize Temp (°C)", "Ramp Rate (°C/min)", "Purge Gas",
               "AMT:Sucrose Ratio", "Hold Time (hr)", "Flow Rate (sccm)"]

# The six model input columns built from those parameters. Order is pinned here and
# used everywhere; if training and prediction matrices ever disagreed on column order
# the model would silently predict nonsense.
FEATS = ["x_temp_C", "x_ramp_C_min", "x_flow_sccm",
         "x_log_hold", "x_log2_ratio", "x_gas_H2"]

dataset: ./dataset/Metal_Sucrose_DesignSpace_fr_T1_encoded.xlsx


## 1. Campaign log — **the only cell you edit**

One tuple per completed run:

```
(run_id, temp_C, ramp_C_min, gas, ratio, hold_hr, flow_sccm, MoC_wt%, esd_wt%, GOF, closure_gap_wt%)
 └──────────── the 6 synthesis parameters → model INPUT ───────────┘ └OBJECTIVE┘ └─ trust in it ─┘
```

The first six become `X`. The seventh is `y`. The last three never enter `X` — they only
set how wide the error bar is on that one observation.

`closure_gap_wt%` is `100 −` (sum of *all* refined phase fractions), normally 0. If your
phases sum to 96.5, enter 3.5: that unquantified mass becomes extra uncertainty rather
than being silently renormalized into MoC.

**Leave out runs you don't trust.** A GOF of 10 means the model doesn't describe the
pattern, so its weight fractions aren't measuring what you think. Omitting is safer than
down-weighting, because a wrong number with a wide error bar still pulls the posterior.

In [2]:
RESULTS = [
    # run    temp ramp  gas  ratio hold flow   MoC   esd    GOF  gap
    ("M7",    800,  15, "N2",  1.0,  0.5,  30,  72.1,  1.50, 0.517, 0.0),
    ("M12",   600,  15, "H2",  1.0,  2.0,  30,  83.8,  5.67, 0.548, 0.0),
    ("M13",   900,  15, "H2",  1.0,  2.0,  30,  23.4,  1.50, 0.932, 0.0),

    # ── append the run you just finished below this line, then re-run all cells ──
]

# Turn the list of tuples into a dataframe whose column names match DESIGN_KEYS exactly,
# so build_features() below can read from it the same way it reads the design space.
log = pd.DataFrame(RESULTS,
                   columns=["run_id"] + DESIGN_KEYS + ["MoC_wt%", "esd_wt%", "GOF", "gap_wt%"])

n_obs = len(log)                                  # used throughout to gate behaviour
print(f"{n_obs} measured runs — round {n_obs + 1} will propose 1 experiment\n")
print(log.to_string(index=False))

3 measured runs — round 4 will propose 1 experiment

run_id  Carburize Temp (°C)  Ramp Rate (°C/min) Purge Gas  AMT:Sucrose Ratio  Hold Time (hr)  Flow Rate (sccm)  MoC_wt%  esd_wt%   GOF  gap_wt%
    M7                  800                  15        N2                1.0             0.5                30     72.1     1.50 0.517      0.0
   M12                  600                  15        H2                1.0             2.0                30     83.8     5.67 0.548      0.0
   M13                  900                  15        H2                1.0             2.0                30     23.4     1.50 0.932      0.0


## 2. Encoding the synthesis parameters

Nine one-hot columns collapsed to six real ones:

| was | now | why |
|---|---|---|
| `feat_ratio_0.5 / 1.0 / 2.0` | `x_log2_ratio` | Ratio is ordinal *and* metric. One-hot makes 0.5↔2.0 exactly as distant as 0.5↔1.0, destroying the ordering — and a level never run is a column that never varies, whose lengthscale has no likelihood gradient at all. |
| `feat_gas_H2` + `feat_gas_N2` | `x_gas_H2` | Two columns for one bit means two lengthscales for one degree of freedom, leaving a flat ridge in the likelihood. For a two-level factor a binary indicator is mathematically equivalent to a categorical kernel. |
| `feat_hold_hr` | `x_log_hold` | Levels 0.5/2/5/10 are geometric — 0.5→2 is 4×, 5→10 is only 2×. A log makes distance match that. |
| temp, ramp, flow | unchanged | Already continuous on physically meaningful scales. |

The rule isn't "discrete → one-hot." It's **one-hot only when the distance between two
values is meaningless.** Temperature has 36 grid levels but 673 °C is a real temperature;
H₂ and N₂ have nothing between them.

In [3]:
def build_features(df):
    """Six synthesis parameters -> six model input columns.

    Reads ONLY from DESIGN_KEYS. It has no access to MoC, esd, GOF or the gap —
    those are labels, not features. The structural test for whether something
    belongs here: can you state its value for an experiment you haven't run yet?
    Temperature, yes (you choose it). GOF, no (it doesn't exist until after XRD).
    """
    f = pd.DataFrame(index=df.index)
    f["x_temp_C"]     = df[DESIGN_KEYS[0]].astype(float)           # 550-900, linear
    f["x_ramp_C_min"] = df[DESIGN_KEYS[1]].astype(float)           # 5/10/15, linear
    f["x_flow_sccm"]  = df[DESIGN_KEYS[5]].astype(float)           # 30/60/100, linear
    f["x_log_hold"]   = np.log(df[DESIGN_KEYS[4]].astype(float))   # 0.5/2/5/10 -> geometric
    f["x_log2_ratio"] = np.log2(df[DESIGN_KEYS[3]].astype(float))  # 0.5/1/2 -> -1/0/+1
    f["x_gas_H2"]     = (df[DESIGN_KEYS[2]] == "H2").astype(float) # the one true categorical
    return f[FEATS]                                                # pin column ORDER


def unit_cube(f, bounds):
    """Scale every column to [0,1] so all six lengthscales start comparable.

    Without this, temperature spans 350 units while the gas indicator spans 1, and
    the optimiser begins from a badly lopsided place. It also makes a fitted
    lengthscale interpretable: 0.61 on the cube = 0.61 x 350 C = ~210 C.
    """
    return np.column_stack([(f[c] - bounds[c][0]) / (bounds[c][1] - bounds[c][0])
                            for c in FEATS])


# Load every candidate experiment. drop_duplicates guards against the same design
# point appearing twice, which would make two grid rows indistinguishable.
space = pd.read_excel(DATA, sheet_name="Design Space")
space = space.drop_duplicates(subset=DESIGN_KEYS).reset_index(drop=True)

feats_all = build_features(space)      # features for all 7,776 candidates

# CRITICAL: bounds come from the FULL design space, never from the measured subset.
# Several measured columns are constant (all three seeds share ramp 15), so training-set
# bounds would divide by zero. And they would shift every round as you add runs, making
# a lengthscale of 0.6 mean something different in round 2 than in round 1.
BOUNDS = {c: (feats_all[c].min(), feats_all[c].max()) for c in FEATS}

X_grid  = unit_cube(feats_all, BOUNDS)                    # (7776, 6) candidates
X_train = unit_cube(build_features(log), BOUNDS)          # (n_obs, 6) measured

print(f"design space : {X_grid.shape[0]:,} candidates x {X_grid.shape[1]} features")
print(f"measured     : {X_train.shape[0]} runs")

# A feature that takes one value across every measured run carries zero information.
# Flagging it here is the honest summary of what the campaign can and cannot answer.
CONST = [FEATS[j] for j in range(len(FEATS)) if len(np.unique(X_train[:, j])) == 1]
if CONST:
    print("\n⚠️  constant across every run so far — the model cannot learn these yet:")
    print("      " + ", ".join(c[2:] for c in CONST))
    print("    Any effect you think you see could belong to one of these instead.")

design space : 7,776 candidates x 6 features
measured     : 3 runs

⚠️  constant across every run so far — the model cannot learn these yet:
      ramp_C_min, flow_sccm, log2_ratio
    Any effect you think you see could belong to one of these instead.


## 3. The objective and its uncertainty

MoC weight fraction goes onto a **logit** scale so the model physically cannot predict
below 0 or above 100 wt%. Predictions are transformed back for display.

The measurement uncertainty travels with it through five steps, each documented inline.

In [4]:
def to_logit(wt, sd_wt):
    """wt% and its standard deviation -> logit value and its VARIANCE.

    p is the fraction, padded by DELTA at both ends so that wt% = 0 or 100 maps to a
    finite number instead of -inf / +inf.

    The variance comes from the delta method: if y = logit(p) then dy/dp = 1/(p(1-p)),
    so var(y) ~= var(p) / (p(1-p))^2. Note this inflates the uncertainty near the
    boundaries, which is correct — a 1 wt% error matters far more at 99% than at 50%.
    """
    p  = (np.asarray(wt, float) + DELTA) / (100.0 + 2 * DELTA)      # pad into (0,1)
    sp = np.asarray(sd_wt, float) / (100.0 + 2 * DELTA)             # same scaling for sd
    return np.log(p / (1 - p)), (sp / (p * (1 - p))) ** 2


def from_logit(z):
    """Inverse of to_logit's value transform: logit -> wt%. Used for every display."""
    return 1.0 / (1.0 + np.exp(-z)) * (100.0 + 2 * DELTA) - DELTA


# ── step 1: start from the reported Rietveld esd ──────────────────────────────
sd_eff = log["esd_wt%"].to_numpy(float)

# ── step 2: inflate by GOF where the refinement fitted poorly ─────────────────
# np.maximum(GOF, 1.0) means a better-than-expected fit (GOF < 1) is never used to
# SHRINK the esd — that would overstate confidence.
sd_eff = sd_eff * np.maximum(log["GOF"].to_numpy(float), 1.0) ** GOF_POWER

# ── step 3: floor it, so a reported esd of 0.00 cannot dominate the fit ───────
sd_eff = np.maximum(sd_eff, SIGMA_FLOOR)

# ── step 4: add the closure gap in quadrature ─────────────────────────────────
# If phases summed to 96.5, the true MoC is somewhere between 96.5 (the rest is
# amorphous) and 100 (the rest is a normalisation artifact). You cannot tell which,
# so the point is marked less certain rather than silently renormalised.
sd_eff = np.sqrt(sd_eff ** 2 + log["gap_wt%"].to_numpy(float) ** 2)

# ── step 5: carry value and uncertainty onto the logit scale ──────────────────
y_raw, y_var_raw = to_logit(log["MoC_wt%"], sd_eff)

# Standardise y to mean 0, sd 1. The priors in section 4 are written for standardised
# outputs, so this must happen before fitting. The noise is divided by the SAME
# variance so it stays in the same units — forgetting this is a classic silent bug
# that makes the model ignore your esds entirely.
Y_MEAN = y_raw.mean()
Y_STD  = max(y_raw.std(ddof=1), 1e-6)            # guard against a single-run campaign
y      = (y_raw - Y_MEAN) / Y_STD                # standardised objective
y_var  = y_var_raw / Y_STD ** 2                  # standardised observation variance

print(pd.DataFrame({
    "run":       log["run_id"],
    "MoC wt%":   log["MoC_wt%"],
    "esd raw":   log["esd_wt%"],
    "GOF":       log["GOF"],
    "esd used":  sd_eff.round(2),                # after GOF inflation + floor + gap
    "y (logit)": y_raw.round(3),
}).to_string(index=False))

run  MoC wt%  esd raw   GOF  esd used  y (logit)
 M7     72.1     1.50 0.517      1.50      0.935
M12     83.8     5.67 0.548      5.67      1.612
M13     23.4     1.50 0.932      1.50     -1.167


## 4. The Gaussian process

`LS_MU = log(sqrt(d))` is a dimension-scaled lengthscale prior (Hvarfner et al., 2024):
expect longer lengthscales in higher dimensions, which stops prior samples from
oscillating implausibly. Section 5 checks that this prior is actually sane for
carburization rather than taking it on faith.

`sample_posterior` is random-walk Metropolis over the log-hyperparameters. It's the
readable stand-in for NUTS — swap in `SaasFullyBayesianSingleTaskGP` +
`fit_fully_bayesian_model_nuts` when you want the production version.

In [5]:
D     = len(FEATS)              # 6 input dimensions
LS_MU = np.log(np.sqrt(D))      # prior MEAN of log-lengthscale, scaled by dimension
LS_SD = 0.75                    # prior SD of log-lengthscale (weakly informative)
SF_MU = 0.0                     # prior mean of log signal variance; 0 because y is
SF_SD = 1.0                     # standardised, so a variance near 1 is expected


def matern52(A, B, ls, sf2):
    """Matern-5/2 covariance between every row of A and every row of B.

    ls is a VECTOR (one lengthscale per input) — that is what makes this ARD, and
    it's how the model can decide temperature matters and ramp rate doesn't.
    """
    d2 = (((A[:, None, :] - B[None, :, :]) / ls) ** 2).sum(-1)   # scaled sq. distance
    r  = np.sqrt(5 * np.maximum(d2, 1e-12))                      # clip guards sqrt(0)
    return sf2 * (1 + r + r ** 2 / 3) * np.exp(-r)               # the 5/2 form


def log_prior(th):
    """Log density of the hyperparameter priors. th = [log ls_1..ls_D, log sf2]."""
    return (-0.5 * (((th[:D] - LS_MU) / LS_SD) ** 2).sum()       # lengthscales
            - 0.5 * ((th[D] - SF_MU) / SF_SD) ** 2)              # signal variance


def log_marglik(th, X, yy, noise):
    """Log marginal likelihood: how well these hyperparameters explain the data.

    `noise` is the per-observation variance from section 3, added to the diagonal.
    This is where your esds actually enter the model — points with poor refinements
    get a bigger diagonal entry and therefore pull the surface less.
    """
    K = matern52(X, X, np.exp(th[:D]), np.exp(th[D]))            # covariance
    K = K + np.diag(noise) + 1e-8 * np.eye(len(yy))              # + noise + jitter
    try:
        L = np.linalg.cholesky(K)                                # stable inverse route
    except np.linalg.LinAlgError:
        return -np.inf                                           # reject this proposal
    a = np.linalg.solve(L.T, np.linalg.solve(L, yy))             # K^-1 y
    return -0.5 * yy @ a - np.log(np.diag(L)).sum() - 0.5 * len(yy) * np.log(2 * np.pi)


def sample_posterior(X, yy, noise, n=4000, burn=1000, seed=0):
    """Draw hyperparameter samples by random-walk Metropolis.

    Why sample instead of optimise: with a handful of points the marginal likelihood
    is nearly flat, so the single "best" lengthscale is barely better than a hundred
    others. Picking one and proceeding as if it were certain is what produces error
    bars roughly half as wide as they should be.
    """
    rng = np.random.default_rng(seed)
    th  = np.concatenate([np.full(D, LS_MU), [SF_MU]])           # start at prior mean
    lp  = log_marglik(th, X, yy, noise) + log_prior(th)          # its log posterior
    out, step, acc = [], 0.3, 0
    for i in range(n + burn):
        prop = th + step * rng.standard_normal(D + 1)            # symmetric proposal
        lp_p = log_marglik(prop, X, yy, noise) + log_prior(prop)
        if np.log(rng.random()) < lp_p - lp:                     # Metropolis accept
            th, lp, acc = prop, lp_p, acc + 1
        if i == burn - 1:                                        # one-shot step tuning
            step *= float(np.clip((acc / burn) / 0.3, 0.3, 3.0)) # target ~30% accept
        if i >= burn:
            out.append(th.copy())                                # keep post-burn draws
    return np.array(out), acc / (n + burn)


def predict(samples, X, yy, noise, Xs, thin=20):
    """Posterior predictive at Xs, averaged over hyperparameter samples.

    Returns the LATENT function mean and sd (no observation noise added) — that is
    what the acquisition function wants, because you are optimising the true MoC
    yield, not a noisy measurement of it. Section 7 adds measurement noise back on
    for the calibration check, which asks a different question.
    """
    mus, vs = [], []
    for th in samples[::thin]:                                   # thin for speed
        ls, sf2 = np.exp(th[:D]), np.exp(th[D])
        K  = matern52(X, X, ls, sf2) + np.diag(noise) + 1e-8 * np.eye(len(yy))
        L  = np.linalg.cholesky(K)
        a  = np.linalg.solve(L.T, np.linalg.solve(L, yy))
        Ks = matern52(Xs, X, ls, sf2)                            # cross-covariance
        v  = np.linalg.solve(L, Ks.T)
        mus.append(Ks @ a)                                       # mean for this draw
        vs.append(np.maximum(sf2 - (v ** 2).sum(0), 1e-12))      # variance for this draw
    mus, vs = np.array(mus), np.array(vs)
    # Law of total variance: average variance WITHIN draws + variance BETWEEN draws.
    # The second term is the hyperparameter uncertainty a point estimate throws away.
    return mus.mean(0), np.sqrt(vs.mean(0) + mus.var(0))

## 5. Prior predictive check

With a handful of points you cannot *measure* whether your error bars are honest, so
you check that the starting assumptions are sane instead. Draw whole curves from the
prior along the temperature axis and ask whether they look like plausible MoC(T).

Carburization yield should be monotone or single-peaked — **0 to 2 turning points**. A
prior generating curves that wiggle five times across 550–900 °C is chemically absurd,
and you would never catch it from the posterior at this sample size.

In [6]:
def prior_predictive(ls_mu, ls_sd, n=600, seed=1):
    """Sample functions from the PRIOR (no data) and summarise their shape."""
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 1, 36)[:, None]                     # 36 temperatures, unit cube
    turns, spans = [], []
    for _ in range(n):
        ls  = np.exp(rng.normal(ls_mu, ls_sd))             # draw a lengthscale
        sf2 = np.exp(rng.normal(SF_MU, SF_SD))             # draw a signal variance
        K   = matern52(t, t, np.array([ls]), sf2) + 1e-8 * np.eye(36)
        f   = np.linalg.cholesky(K) @ rng.standard_normal(36)   # draw a whole curve
        w   = from_logit(Y_MEAN + Y_STD * f)               # put it back in wt%
        # count sign changes of the slope = how many times the curve turns around
        turns.append(int((np.diff(np.sign(np.diff(w))) != 0).sum()))
        spans.append(w.max() - w.min())                    # how much range it implies
    return np.array(turns), np.array(spans)


print("prior predictive along the temperature axis (550-900 °C)\n")
print(f"{'prior on log-lengthscale':<34} {'turning pts (med, p90)':<24} {'wt% span (med)'}")
for name, mu in [("too short  LogN(-1.5, 0.75)", -1.5),
                 (f"in use     LogN({LS_MU:.2f}, {LS_SD})", LS_MU),
                 ("too long   LogN( 2.5, 0.75)", 2.5)]:
    t, s = prior_predictive(mu, LS_SD)
    flag = "  <- in use" if abs(mu - LS_MU) < 1e-9 else ""
    print(f"{name:<34} {np.median(t):>6.0f}, {np.percentile(t, 90):<16.0f} "
          f"{np.median(s):>10.0f}{flag}")
print("\ntarget: 0-2 turning points. More means the prior expects physical nonsense.")

prior predictive along the temperature axis (550-900 °C)

prior on log-lengthscale           turning pts (med, p90)   wt% span (med)
too short  LogN(-1.5, 0.75)             5, 10                       53
in use     LogN(0.90, 0.75)             0, 2                         7  <- in use
too long   LogN( 2.5, 0.75)             0, 4                         1

target: 0-2 turning points. More means the prior expects physical nonsense.


## 6. Fit, and what the data has actually taught the model

In [7]:
# Draw hyperparameter samples conditioned on every measured run so far.
samples, acc = sample_posterior(X_train, y, y_var, seed=RANDOM_SEED)
print(f"MCMC acceptance {acc:.2f} ({len(samples)} samples kept)")
if not 0.15 < acc < 0.60:
    print("⚠️  acceptance outside 0.15-0.60 — chain may be mixing poorly")

print("\nwhat the data has actually taught the model:\n")
print(f"{'feature':<14} {'varies':<8} {'post. mean log-ls':<20} {'shift from prior'}")
for j, c in enumerate(FEATS):
    post   = samples[:, j].mean()                      # posterior mean log-lengthscale
    shift  = (post - LS_MU) / LS_SD                    # how far it moved, in prior SDs
    varies = len(np.unique(X_train[:, j])) > 1         # did this feature ever change?
    # A feature that never varies produces no likelihood gradient, so its posterior
    # is EXACTLY its prior. That is correct behaviour, not a bug.
    note = ("learned" if abs(shift) > 0.5
            else ("— posterior IS prior" if not varies else "weak"))
    print(f"{c[2:]:<14} {str(varies):<8} {post:>12.2f}      {shift:>+6.2f} prior SD   {note}")

print("\nShorter lengthscale = more influential. A value pinned at the prior means")
print("that feature has taught the model nothing yet.")

MCMC acceptance 0.35 (4000 samples kept)

what the data has actually taught the model:

feature        varies   post. mean log-ls    shift from prior
temp_C         True            -0.05       -1.26 prior SD   learned
ramp_C_min     False            0.85       -0.06 prior SD   — posterior IS prior
flow_sccm      False            0.99       +0.13 prior SD   — posterior IS prior
log_hold       True             0.87       -0.04 prior SD   weak
log2_ratio     False            0.83       -0.09 prior SD   — posterior IS prior
gas_H2         True             0.78       -0.16 prior SD   weak

Shorter lengthscale = more influential. A value pinned at the prior means
that feature has taught the model nothing yet.


## 7. Calibration check

Hold out one run, refit without it, predict it, and standardise the error by the full
predictive standard deviation:

```
z = (actual − predicted) / sqrt(posterior_var + measurement_var)
```

If the error bars are honest, those z-values behave like draws from a standard normal:
**SD(z) ≈ 1**, mean near 0, about 95% inside ±2. Above 1 means overconfident, below
means overcautious.

This matters more than accuracy in BO. RMSE grades the mean; calibration grades the
*uncertainty*, and the uncertainty is what the acquisition function spends your furnace
time on. Below ~6 runs the cell refuses to report — holding one out of three leaves two,
which determines nothing.

In [8]:
def loo_z(X, yy, noise, seed=0):
    """Leave-one-out standardised residuals, refitting hyperparameters each fold.

    Refitting matters: reusing hyperparameters fitted on ALL the data leaks the
    held-out point into the model and makes the result optimistically biased.
    """
    zs = []
    for i in range(len(yy)):
        m = np.ones(len(yy), bool)
        m[i] = False                                           # drop point i
        s, _  = sample_posterior(X[m], yy[m], noise[m], n=1200, burn=400, seed=seed)
        mu, sd = predict(s, X[m], yy[m], noise[m], X[i:i + 1])  # predict it
        # Denominator = posterior (epistemic) + measurement (aleatoric). Both belong
        # here because we are asking whether this NOISY OBSERVATION is plausible.
        zs.append((yy[i] - mu[0]) / np.sqrt(sd[0] ** 2 + noise[i]))
    return np.array(zs)


if n_obs < 6:
    print(f"⏸  {n_obs} runs — skipping.")
    print("    Leave-one-out below ~6 points measures the futility of fitting on n-1,")
    print("    not the quality of the model. Becomes meaningful around 10-15 runs.")
else:
    z = loo_z(X_train, y, y_var, seed=RANDOM_SEED)
    print("z per run:", np.round(z, 2))
    print(f"\nSD(z)   = {z.std():.2f}   (want ~1; >1 overconfident, <1 overcautious)")
    print(f"mean(z) = {z.mean():+.2f}   (want ~0; otherwise systematic bias)")
    print(f"|z| < 2 : {int((abs(z) < 2).sum())}/{n_obs}   (want ~95%)")
    # An SD estimated from n numbers has this much relative error. At n=8 it is ~27%,
    # so 1.77 vs 1.0 is a real signal but 1.77 vs 2.2 is noise.
    print(f"\nrel. standard error on that SD ~ {1 / np.sqrt(2 * (n_obs - 1)):.0%}")
    if n_obs < 10:
        print("still thin — read the individual z values, not the summary.")
    # |z| > 4 happens about once in 22,000 for a calibrated model. Two of them in
    # eight runs is not bad luck, it is proof the error bars are wrong there.
    bad = np.where(np.abs(z) > 4)[0]
    if len(bad):
        print("\n⚠️  |z| > 4 on: " + ", ".join(log["run_id"].iloc[bad]))
        print("    A calibrated model does that ~1 in 22,000 times. The model is blind")
        print("    in whatever regime those runs occupy.")

⏸  3 runs — skipping.
    Leave-one-out below ~6 points measures the futility of fitting on n-1,
    not the quality of the model. Becomes meaningful around 10-15 runs.


## 8. Choose the single next experiment

Two strategies, and which is right depends on how much the model knows.

**Exploration (maximin distance)** ignores the model and picks the candidate furthest
from everything already run. Early on this buys far more than EI, and it breaks the
confounds that otherwise make every effect unattributable.

**Expected improvement** weighs predicted value against uncertainty. It's the right tool
once the model has real structure. Before then it isn't: with almost no data, posterior
variance peaks at the corners of the design box, so EI proposes extremes it knows nothing
about — chasing its own ignorance rather than a real optimum.

The switch happens automatically at `EI_AFTER` runs. Both candidates are printed either
way, so you can override with domain knowledge.

In [9]:
# Posterior over every candidate in the design space.
mu_g, sd_g = predict(samples, X_train, y, y_var, X_grid)

# Back to wt% for display, with a 95% credible interval.
space["pred_MoC_wt%"] = from_logit(Y_MEAN + Y_STD * mu_g)
space["pred_lo95"]    = from_logit(Y_MEAN + Y_STD * (mu_g - 1.96 * sd_g))
space["pred_hi95"]    = from_logit(Y_MEAN + Y_STD * (mu_g + 1.96 * sd_g))

# ── expected improvement, in standardised logit units ─────────────────────────
# EI = E[max(f(x) - best, 0)] under the posterior. The closed form for a Gaussian:
#   imp = mu - best - xi ;  EI = imp * Phi(imp/sd) + sd * phi(imp/sd)
imp = mu_g - y.max() - EI_XI
space["EI"] = np.where(sd_g > 0,
                       imp * norm.cdf(imp / sd_g) + sd_g * norm.pdf(imp / sd_g),
                       0.0)

# Exclude anything already measured — rounding guards float comparison.
measured = set(map(tuple, np.round(X_train, 9)))
fresh    = np.array([tuple(r) not in measured for r in np.round(X_grid, 9)])
print(f"{fresh.sum():,} unmeasured candidates\n")

# ── candidate A: expected improvement ─────────────────────────────────────────
idx_ei = int(space.loc[fresh, "EI"].idxmax())

# ── candidate B: exploration — furthest point from everything measured ────────
# For each fresh candidate, distance to its NEAREST measured run; take the maximum.
d_near = np.full(len(X_grid), -np.inf)
d_near[fresh] = np.linalg.norm(X_grid[fresh][:, None, :] - X_train[None, :, :],
                               axis=2).min(axis=1)
idx_explore = int(np.argmax(d_near))

# ── pick one ──────────────────────────────────────────────────────────────────
USE_EI = n_obs >= EI_AFTER
CHOSEN = idx_ei if USE_EI else idx_explore
reason = (f"{n_obs} runs >= {EI_AFTER} — the model has structure worth exploiting"
          if USE_EI else
          f"only {n_obs} runs — exploring first; EI would chase prior variance at the corners")

print(f"strategy: {'expected improvement' if USE_EI else 'exploration'}")
print(f"          ({reason})\n")

cols = DESIGN_KEYS + ["pred_MoC_wt%", "pred_lo95", "pred_hi95", "EI"]
cmp_ = space.loc[[CHOSEN, idx_ei if not USE_EI else idx_explore], cols].round(2)
cmp_.insert(0, "", ["→ CHOSEN", "  (other strategy)"])
print(cmp_.to_string(index=False))

# Report which currently-constant features this suggestion moves — the main thing
# a single early experiment can buy you.
if CONST:
    breaks = [c[2:] for c in CONST
              if X_grid[CHOSEN, FEATS.index(c)] != X_train[0, FEATS.index(c)]]
    print(f"\nbreaks these confounds: {', '.join(breaks) if breaks else 'none'}")

7,773 unmeasured candidates

strategy: exploration
          (only 3 runs — exploring first; EI would chase prior variance at the corners)

                    Carburize Temp (°C)  Ramp Rate (°C/min) Purge Gas  AMT:Sucrose Ratio  Hold Time (hr)  Flow Rate (sccm)  pred_MoC_wt%  pred_lo95  pred_hi95   EI
          → CHOSEN                  550                   5        N2                0.5            10.0               100         74.40       5.32      99.88 0.36
  (other strategy)                  550                   5        N2                0.5             0.5                30         80.16      10.79      99.83 0.40

breaks these confounds: ramp_C_min, flow_sccm, log2_ratio


## 9. ➡️  Run this experiment

In [10]:
r = space.loc[CHOSEN]                                     # the single chosen candidate

print("=" * 74)
print(f"  ROUND {n_obs + 1}  —  ONE EXPERIMENT")
print("=" * 74)
print(f"  temperature  {r[DESIGN_KEYS[0]]:>6.0f} °C        ramp   {r[DESIGN_KEYS[1]]:>4.0f} °C/min")
print(f"  hold         {r[DESIGN_KEYS[4]]:>6.1f} hr        flow   {r[DESIGN_KEYS[5]]:>4.0f} sccm")
print(f"  gas          {r[DESIGN_KEYS[2]]:>6}           AMT:sucrose  {r[DESIGN_KEYS[3]]}")
print("-" * 74)
print(f"  model expects {r['pred_MoC_wt%']:.0f} wt% MoC"
      f"   (95% interval {r['pred_lo95']:.0f}-{r['pred_hi95']:.0f})")
print("=" * 74)
print()
# textwrap keeps decimals intact — splitting on "." would break "0.6 g" into "0." / "6 g"
print(textwrap.fill(str(r["Procedure"]), width=70,
                    initial_indent="   ", subsequent_indent="   "))

print("\n" + "=" * 74)
print("  When it's done, paste this line into RESULTS and re-run all cells:")
print("=" * 74)
print(f'    ("R{n_obs + 1}", {r[DESIGN_KEYS[0]]:.0f}, {r[DESIGN_KEYS[1]]:.0f}, '
      f'"{r[DESIGN_KEYS[2]]}", {r[DESIGN_KEYS[3]]}, {r[DESIGN_KEYS[4]]}, '
      f'{r[DESIGN_KEYS[5]]:.0f},  ???, ???, ???, 0.0),')
print("\n    fill in:  MoC_wt%,  esd_wt%,  GOF,  closure_gap_wt%")

  ROUND 4  —  ONE EXPERIMENT
  temperature     550 °C        ramp      5 °C/min
  hold           10.0 hr        flow    100 sccm
  gas              N2           AMT:sucrose  0.5
--------------------------------------------------------------------------
  model expects 74 wt% MoC   (95% interval 5-100)

   1 g of ammonium heptamolybdate tetrahydrate dissolved in 4 mL of DI
   water. Separately, 2 g of sucrose dissolved in 3.5 mL of DI water
   at 50 °C. The two solutions were then combined. The combined
   solution was dried at 120 °C in air for 24 hours, then crushed and
   sieved to 212 microns. 0.6 g of the sieved precursor was carburized
   at a ramp rate of 5 °C/min to 550 °C for 10 hr under 100 sccm of N2
   gas. After carburization the sample was cooled under 30 sccm N2 and
   passivated under 30 sccm 1% O2/N2 for 2 hours.

  When it's done, paste this line into RESULTS and re-run all cells:
    ("R4", 550, 5, "N2", 0.5, 10.0, 100,  ???, ???, ???, 0.0),

    fill in:  MoC_wt%,  e

## 10. Campaign state

In [11]:
# Measured runs plus the one pending suggestion, in a single table.
board = log[["run_id"] + DESIGN_KEYS + ["MoC_wt%"]].copy()
board["status"] = "measured"

nxt = space.loc[[CHOSEN], DESIGN_KEYS].copy()
nxt.insert(0, "run_id", f"R{n_obs + 1}")
nxt["MoC_wt%"] = np.nan                                   # not yet known
nxt["status"]  = "proposed"
board = pd.concat([board, nxt], ignore_index=True)


def grey_pending(row):
    """Render the un-run suggestion in grey italic so it can't be mistaken for data."""
    return ["color:#999; font-style:italic" if row["status"] == "proposed" else ""
            for _ in row]


def bar(v):
    """Inline bar chart in the MoC column: fill proportional to weight fraction."""
    if pd.isna(v):
        return "color:#bbb"
    return (f"background: linear-gradient(90deg,#4e91d2 {float(v):.0f}%,"
            f"transparent {float(v):.0f}%); font-weight:bold")


try:
    display(board.style
            .apply(grey_pending, axis=1)
            .map(bar, subset=["MoC_wt%"])
            .format({"MoC_wt%": lambda v: f"{v:.1f}" if pd.notna(v) else "⏳ pending"})
            .set_caption("Cubic MoC campaign — sequential GP BO")
            .hide(axis="index"))
except Exception:
    print(board.to_string(index=False))                   # fallback outside Jupyter

best = log.loc[log["MoC_wt%"].idxmax()]
print(f"\n🏆 best measured: {best['MoC_wt%']:.1f} wt% MoC  ({best['run_id']} — "
      f"{best[DESIGN_KEYS[0]]:.0f} °C, {best[DESIGN_KEYS[2]]}, {best[DESIGN_KEYS[5]]:.0f} sccm)")

# The model's optimistic guess anywhere it hasn't measured. Read the interval, not
# the mean — a narrow interval this early would mean the model was lying to you.
top = space.loc[fresh].nlargest(1, "pred_MoC_wt%").iloc[0]
print(f"📈 model's best guess, unmeasured: {top['pred_MoC_wt%']:.0f} wt% at "
      f"{top[DESIGN_KEYS[0]]:.0f} °C, {top[DESIGN_KEYS[2]]}, {top[DESIGN_KEYS[5]]:.0f} sccm "
      f"(95% interval {top['pred_lo95']:.0f}-{top['pred_hi95']:.0f})")

run_id,Carburize Temp (°C),Ramp Rate (°C/min),Purge Gas,AMT:Sucrose Ratio,Hold Time (hr),Flow Rate (sccm),MoC_wt%,status
M7,800,15,N2,1.000000,0.500000,30,72.1,measured
M12,600,15,H2,1.000000,2.000000,30,83.8,measured
M13,900,15,H2,1.000000,2.000000,30,23.4,measured
R4,550,5,N2,0.500000,10.000000,100,⏳ pending,proposed



🏆 best measured: 83.8 wt% MoC  (M12 — 600 °C, H2, 30 sccm)
📈 model's best guess, unmeasured: 85 wt% at 560 °C, N2, 30 sccm (95% interval 34-99)


## Notes

**Read the interval, not the mean.** At this sample size the 95% bands are wide on
purpose. Narrow bands would mean the model was lying, and a lying model makes the
acquisition stop exploring — which is how a campaign converges early onto a local
optimum and stays there.

**Break the confounds first.** Section 2 lists features constant across every measured
run. Until they vary, their fitted lengthscales mean nothing, and any effect you think
you've found could belong to a variable you never moved.

**Bad refinements are worse than missing data.** GOF ≈ 10 means the structural model
doesn't explain the pattern. Leave those runs out of `RESULTS`.

**Why one at a time.** Each result reshapes the posterior everywhere. A batch of four
is chosen by a model that has seen none of them, so the picks tend to cluster and
re-ask one question four times. If you ever do need a batch — parallel furnaces, a
shared calcination — use one that penalises picks for being near each other rather
than taking the top four by acquisition value.

**Production upgrade.** This maps directly onto BoTorch: `SaasFullyBayesianSingleTaskGP`
with `train_Yvar` for the fixed esds, `fit_fully_bayesian_model_nuts` for sampling,
`LogExpectedImprovement` with `optimize_acqf_discrete` over `X_grid`. Same structure,
better samplers, and gradient-based optimisation if you stop restricting yourself to
the 36 preset temperatures — your furnace can be set to 673 °C, and that grid is an
artifact of how the space was enumerated, not a physical constraint.